# Objective

The objective of this notebook is to build a **Recurrent Neural Network (RNN)** for text classification using TensorFlow and the Hugging Face dataset. The notebook demonstrates the complete Natural Language Processing (NLP) pipeline, including loading the dataset, cleaning and normalizing text, tokenizing words into numerical sequences, padding sequences to a fixed length, encoding class labels, designing an RNN architecture with Embedding, SimpleRNN, and Dense layers, training the model, and evaluating its performance using accuracy and validation accuracy metrics.



# Workflow

- Load a text classification dataset from Hugging Face.
- Clean and normalize the text data.
- Convert text into numerical sequences using tokenization.
- Pad and truncate sequences to a fixed length.
- Encode class labels using one-hot encoding.
- Build an RNN model using Embedding, SimpleRNN, and Dense layers.
- Train the model using the Adam optimizer and categorical cross-entropy loss.
- Evaluate the model using accuracy and validation accuracy metrics.

## Step 1: Install and Import Required Libraries
**Objective:**  
To install and import all the necessary libraries required for loading the dataset, preprocessing text, building the RNN model, training it, and evaluating its performance.

---

In [2]:
# Install the required libraries (Run this only once)
# !pip install datasets tensorflow scikit-learn

# Import libraries
import re
import numpy as np

# Load dataset from Hugging Face
from datasets import load_dataset

# TensorFlow imports
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

# Train-Test Split
from sklearn.model_selection import train_test_split

## Step 2: Load the Dataset
**Objective:**  
To load the AG News dataset from Hugging Face into memory, providing text samples and their corresponding class labels for text classification.

In [3]:
# Load AG News dataset directly from Hugging Face
dataset = load_dataset("wangrongsheng/AG_News")

# Display dataset information
print(dataset)

# Display first training sample
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


## Step 3: Extract Text and Labels
**Objective:**  
To separate the text data (input features) and class labels (target variable) from the dataset so they can be used for preprocessing and model training.

In [4]:
# Extract training text

texts = dataset["train"]["text"]

# Extract labels
labels = dataset["train"]["label"]

print("Total Articles:", len(texts))
print("First Article:")
print(texts[0])

print("Label:", labels[0])

Total Articles: 120000
First Article:
Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
Label: 2


## Step 4: Clean and Normalize the Text
**Objective:**  
To preprocess the text by converting it to lowercase, removing punctuation, numbers, special characters, and extra spaces, ensuring consistent and clean input for the model.

In [5]:
# Function for cleaning text

def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove numbers and punctuation
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


# Apply cleaning to every article
cleaned_texts = []

for text in texts:
    cleaned_texts.append(clean_text(text))

# Print sample
print(cleaned_texts[0])

wall st bears claw back into the black reuters reuters shortsellers wall streets dwindlingband of ultracynics are seeing green again


## Step 5: Tokenize the Text
**Objective:**  
To convert the cleaned text into numerical sequences by assigning a unique integer to each word, enabling the neural network to process textual data.

---

In [6]:
# Maximum vocabulary size
vocab_size = 10000

# Create tokenizer
tokenizer = Tokenizer(num_words=vocab_size)

# Learn vocabulary
tokenizer.fit_on_texts(cleaned_texts)

# Convert text into integer sequences
sequences = tokenizer.texts_to_sequences(cleaned_texts)

# Print first sequence
print(sequences[0])

# Print vocabulary size
print("Vocabulary Size:", len(tokenizer.word_index))

[391, 324, 1525, 99, 54, 1, 812, 23, 23, 391, 1988, 4, 34, 3893, 737, 295]
Vocabulary Size: 91343


## Step 6: Pad and Truncate the Sequences
**Objective:**  
To ensure all text sequences have the same length by padding shorter sequences with zeros and truncating longer sequences, making them suitable for batch processing in the RNN.

---


In [7]:
# Every sentence will become exactly 50 words

max_length = 50

# Pad shorter sequences and truncate longer ones

X = pad_sequences(
    sequences,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

print(X.shape)

print(X[0])

(120000, 50)
[ 391  324 1525   99   54    1  812   23   23  391 1988    4   34 3893
  737  295    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0]


## Step 7: One-Hot Encode the Labels
**Objective:**  
To convert the categorical class labels into one-hot encoded vectors, allowing the model to perform multi-class classification using the Softmax output layer.

---


In [8]:
# Convert labels into categorical vectors

y = to_categorical(labels)

print(y.shape)

print(y[0])

(120000, 4)
[0. 0. 1. 0.]


## Step 8: Split the Dataset into Training and Testing Sets
**Objective:**  
To divide the dataset into training and testing subsets so the model can be trained on one portion of the data and evaluated on unseen data.

---


In [9]:
# Split data into train and test

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(96000, 50)
(24000, 50)


## Step 9: Build the Simple RNN Model
**Objective:**  
To design a Sequential RNN architecture using an Embedding layer for word representation, a SimpleRNN layer for learning sequential patterns, and a Dense layer with Softmax activation for classification.

---

In [10]:
# Create Sequential model

model = Sequential()

# Embedding Layer
# Converts integer IDs into dense vectors

model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        input_length=max_length
    )
)

# Simple RNN Layer
# Learns sequential relationships

model.add(
    SimpleRNN(64)
)

# Output Layer
# 4 neurons because AG News has 4 classes

model.add(
    Dense(
        4,
        activation='softmax'
    )
)

# Display model architecture

model.summary()

c:\Users\MOHD NADEEM\OneDrive\Documents\College_ML_and_DL\myenv\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Step 10: Compile the Model
**Objective:**  
To configure the model by selecting the optimizer, loss function, and evaluation metric that will guide the learning process during training.

---

In [11]:
# Compile model

model.compile(

    optimizer='adam',

    loss='categorical_crossentropy',

    metrics=['accuracy']

)

## Step 11: Train the Model
**Objective:**  
To train the RNN model using the training dataset, allowing it to learn patterns in the text by minimizing prediction errors over multiple epochs.

---


In [12]:
# Train model

history = model.fit(

    X_train,

    y_train,

    epochs=5,

    batch_size=32,

    validation_split=0.2

)

Epoch 1/5
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 24s 9ms/step - accuracy: 0.3871 - loss: 1.2678 - val_accuracy: 0.6600 - val_loss: 0.9563
Epoch 2/5
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 24s 10ms/step - accuracy: 0.6213 - loss: 0.9942 - val_accuracy: 0.6556 - val_loss: 0.9309
Epoch 3/5
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 21s 9ms/step - accuracy: 0.6943 - loss: 0.8246 - val_accuracy: 0.7445 - val_loss: 0.7162
Epoch 4/5
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 22s 9ms/step - accuracy: 0.6927 - loss: 0.8120 - val_accuracy: 0.6659 - val_loss: 0.8945
Epoch 5/5
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 28s 12ms/step - accuracy: 0.7091 - loss: 0.7942 - val_accuracy: 0.6657 - val_loss: 0.9100


## Step 12: Evaluate the Model
**Objective:**  
To assess the model's performance on the test dataset by calculating the loss and accuracy, ensuring that the model generalizes well to unseen data.

---


In [13]:
# Evaluate model on test data

loss, accuracy = model.evaluate(X_test, y_test)

print("Test Loss:", loss)

print("Test Accuracy:", accuracy)

750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6690 - loss: 0.9097
Test Loss: 0.9096554517745972
Test Accuracy: 0.6690416932106018
